# CoalGameRec — C1 Matched-Controls Confirmatory Run

**Purpose:** run the C1 confirmatory study (frozen v3 protocol + matched validation-informed controls `valid-sim` / `valid-linear`), stream every milestone to this notebook, and persist the full log to `<run_dir>/run.log` so the reviewer/implementer loop can inspect it.

**How to use (in order):**
1. Run **Cell 1** (setup) and **Cell 2** (environment check). Fix any missing dependency it reports.
2. Run **Cell 3** (the run). Keep this tab open. Milestones appear below the cell; the complete log (incl. progress) is appended to `run.log`.
   - It is **resume-safe**: if the kernel dies or you stop the cell, just re-run Cell 3 — completed seeds are skipped.
   - You can check progress at any time with **Cell 4** (status), even after a kernel restart.
3. When you see `ALL DONE`, run **Cell 5** (analysis + tables) and **Cell 6** (results preview).
4. Commit + push with the commands shown by **Cell 7**.

Estimated time on Apple Silicon (MPS): ~30–40 min per seed × 5 seeds. CPU-only machines are slower.

In [ ]:
# Cell 1 — Setup (paths, dataset selection, device)
from pathlib import Path
import os, sys, platform

# ==== EDIT HERE IF NEEDED ====
DATASET = "ml1m"          # "ml1m" (needed) or "amazon" (already complete; re-run optional)
SEEDS = [42, 43, 44, 45, 46]
# =============================

CWD = Path.cwd().resolve()
CODE_DIR = CWD.parent if CWD.name == "notebooks" else CWD
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

RESULTS = CODE_DIR / "results" / "journal_runs"
SRC_NAME = {"ml1m": "ml1m_lightgcn_v3_prospective", "amazon": "amazon_books_lightgcn_v3_prospective"}[DATASET]
OUT_NAME = {"ml1m": "ml1m_lightgcn_v4_matched_controls", "amazon": "amazon_books_lightgcn_v4_matched_controls"}[DATASET]
SOURCE = RESULTS / SRC_NAME
OUT = RESULTS / OUT_NAME
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / "run.log"

assert SOURCE.exists(), f"source run missing: {SOURCE}"
assert (SOURCE / "splits" / "train.parquet").exists(), f"splits missing in {SOURCE}/splits"

print("CODE_DIR :", CODE_DIR)
print("dataset  :", DATASET)
print("source   :", SOURCE)
print("output   :", OUT)
print("log file :", LOG)
print("seeds    :", SEEDS)

In [ ]:
# Cell 2 — Environment check (device + dependencies)
missing = []
for mod, pkg in [("torch", "torch"), ("numpy", "numpy==1.26.4"), ("scipy", "scipy==1.13.0"),
                 ("pandas", "pandas==2.2.2"), ("pyarrow", "pyarrow==15.0.2"), ("tqdm", "tqdm==4.66.4"),
                 ("yaml", "pyyaml==6.0.1")]:
    try:
        __import__(mod)
    except ImportError:
        missing.append(pkg)
if missing:
    print("MISSING DEPENDENCIES — install with:")
    print("  pip install " + " ".join(f'"{m}"' for m in missing))
    raise SystemExit("install missing dependencies, then re-run this cell")

import torch, numpy, scipy, pandas, platform
if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"device={DEVICE} | torch={torch.__version__} | numpy={numpy.__version__} | scipy={scipy.__version__} | pandas={pandas.__version__}")
print(f"platform={platform.platform()} | python={platform.python_version()}")
print("OK — ready to run Cell 3.")

In [ ]:
# Cell 3 — RUN C1 (streams milestones here; full log -> run.log). Resume-safe.
import subprocess, time

cmd = [sys.executable, str(CODE_DIR / "scripts" / "run_matched_controls.py"),
       "--dataset", DATASET, "--source-run", str(SOURCE), "--out", str(OUT),
       "--seeds", *[str(s) for s in SEEDS]]
env = dict(os.environ, COALGAME_DEVICE=DEVICE)
print("CMD:", " ".join(cmd))
print("DEVICE:", DEVICE)
print("=" * 70)

logf = open(LOG, "a")
logf.write(f"\n===== notebook run started {time.strftime('%Y-%m-%d %H:%M:%S')} device={DEVICE} python={platform.python_version()} =====\n")
logf.flush()

p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=str(CODE_DIR), env=env)
buf = b""
while True:
    ch = p.stdout.read(1)
    if not ch:
        break
    if ch in (b"\n", b"\r"):
        line = buf.decode(errors="replace").strip()
        buf = b""
        if not line:
            continue
        logf.write(line + "\n")
        logf.flush()
        if "it/s" in line or "it]" in line:
            continue  # tqdm progress: log-file only, don't spam the notebook
        print(line, flush=True)
    else:
        buf += ch
p.wait()
logf.write(f"===== notebook run finished {time.strftime('%Y-%m-%d %H:%M:%S')} exit={p.returncode} =====\n")
logf.close()
print("=" * 70)
print("exit code:", p.returncode)
if p.returncode == 0:
    print("SUCCESS — continue to Cell 5 (analysis).")
else:
    print("FAILED or interrupted — re-run this cell to resume (completed seeds are skipped).")

In [ ]:
# Cell 4 — Status check (safe to run any time, even after a kernel restart)
import subprocess
r = subprocess.run(["pgrep", "-af", "run_matched_controls.py"], capture_output=True, text=True)
print("runner process:", r.stdout.strip() if r.stdout.strip() else "NOT RUNNING")
print()
if LOG.exists():
    lines = [l for l in LOG.read_text(errors="replace").splitlines() if l.strip()]
    milestones = [l for l in lines if any(k in l for k in ("epoch 15/15", "seed 4", "ALL DONE", "notebook run"))]
    print(f"log: {LOG} ({len(lines)} lines)")
    print("last milestones:")
    for l in milestones[-8:]:
        print("  ", l)
else:
    print("no log yet — run Cell 3")
print()
for s in SEEDS:
    done = (OUT / "raw" / f"seed_{s}" / "summary_by_family.csv").exists()
    print(f"seed {s}: {'COMPLETE' if done else 'pending'}")

In [ ]:
# Cell 5 — Analysis (run after ALL DONE): paired contrasts + manuscript tables
import subprocess
for cmd in [
    [sys.executable, str(CODE_DIR / "scripts" / "analyze_matched_controls.py"), "--run-dir", str(OUT)],
    [sys.executable, str(CODE_DIR / "scripts" / "make_matched_controls_tables.py")],
]:
    print("RUN:", " ".join(cmd))
    r = subprocess.run(cmd, cwd=str(CODE_DIR))
    assert r.returncode == 0, f"analysis failed with exit {r.returncode}"
print("analysis complete")

In [ ]:
# Cell 6 — Results preview (5-seed means + Holm contrasts)
import pandas as pd
pd.set_option("display.width", 200)
mm = pd.read_csv(OUT / "tables" / "summary_mean_std.csv", header=[0, 1], index_col=[0, 1, 2])
keep = ["HitRate@20", "NDCG@20", "Coverage@20"]
tab = mm[[ (m, s) for m in keep for s in ("mean", "std") ]].round(5)
print("=== 5-seed mean±SD by family ===")
display(tab)
pc = pd.read_csv(OUT / "tables" / "paired_bootstrap_loo_vs_matched_controls.csv")
print("=== LOO vs matched controls (Holm F=12) ===")
display(pc[["contrast", "mean_diff_conditional_user", "ci95_low", "ci95_high", "bootstrap_p_report", "holm_reject_0.05"]])

In [ ]:
# Cell 7 — Hand back to the loop: commit + push (review these commands, then run in a terminal)
print("Run these in a terminal from the repo root:")
print("""
git add paper-ideas/CoalGameRec/code/results/journal_runs/{out_name} \\
        paper-ideas/CoalGameRec/code/notebooks/CoalGameRec_C1_Run.ipynb \\
        paper-ideas/CoalGameRec/manuscript_assets
git commit -m "C1 confirmatory run complete ({ds})"
git push origin arena/019fdd75-next-paper
""".format(out_name=OUT_NAME, ds=DATASET))